# INF201 Deliverable 3: RPG Battle Simulator

GitHub repository: https://github.com/fabianfeiler/inf201-h26-deliverable-3

In [1]:
import random
from battle import Battle
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon

___________________________________
## Class overview

| Class | Module | Purpose | Relationship |
|---|---|---|---|
| `Dice` | `dice.py` | Rolls a random number from 1 to `sides` | Used by `Weapon` |
| `Weapon` | `items.py` | Rolls damage with its `Dice` | Has a `Dice` (composition) |
| `Character` | `character.py` | Base class with HP, damage and attacks | Can have a `Weapon` (composition) |
| `Warrior` | `character.py` | Defensive character with armor | Is a `Character` (inheritance) |
| `Mage` | `character.py` | Uses mana for magic attacks | Is a `Character` (inheritance) |
| `Battle` | `battle.py` | Runs turn-based combat | Has two `Character` objects |

The cell below creates one object of each class and prints its attributes and public methods.

In [2]:
def show(obj: object) -> None:
    """Print the class name, attributes and public methods of an object."""
    attributes = list(vars(obj))
    methods = [name for name in dir(obj) if not name.startswith("_") and name not in attributes]
    print(type(obj).__name__)
    print(f"  Attributes: {attributes}")
    print(f"  Methods and properties: {methods}\n")


sword = Weapon("Sword", Dice(sides=12))
objects = [Dice(), sword, Character("Hero"), Warrior("Warrior", weapon=sword), Mage("Mage"), Battle(Character("A"), Character("B"))]

for obj in objects:
    show(obj)

Dice
  Attributes: ['sides']
  Methods and properties: ['roll']

Weapon
  Attributes: ['name', 'damage_dice']
  Methods and properties: ['roll_damage']

Character
  Attributes: ['name', '_hp', 'hp_max', 'damage', 'weapon']
  Methods and properties: ['attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']

Warrior
  Attributes: ['name', '_hp', 'hp_max', 'damage', 'weapon', 'armor']
  Methods and properties: ['attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']

Mage
  Attributes: ['name', '_hp', 'hp_max', 'damage', 'weapon', 'mana', 'mana_max']
  Methods and properties: ['attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']

Battle
  Attributes: ['hero', 'enemy', 'max_rounds', 'verbose', 'round']
  Methods and properties: ['play_round', 'simulate_battle']



**Result:** `Warrior` and `Mage` have all the methods from `Character` through inheritance, and only add their own attributes (`armor`, `mana`, `mana_max`). `_hp` is listed as an attribute, but is only read through the `hp` property and changed by `take_damage()` and `heal()`. Special methods like `__call__` and `__repr__` in `Dice` are not listed, since names starting with `_` are filtered out. The same goes for `_attack()` in `Battle`, which is a private helper method.

-------
## Task 1: Update the Character class

`attack()` used to change the target's `_hp` directly. Now it calls `target.take_damage()`, so each character only changes its own HP. We also added `heal()` and the property `is_alive`.

In [3]:
enemy = Character("Enemy", hp_max=25)
hero = Character("Hero")

hero.attack(enemy)
hero.attack(enemy)
hero.attack(enemy)
print(enemy.hp, enemy.is_alive)

enemy.heal(100)
print(enemy.hp)


try:
    enemy.is_alive = True
except AttributeError as err:
    print(f"AttributeError: {err}")

0 False
25
AttributeError: property 'is_alive' of 'Character' object has no setter


**Result:** After three attacks the enemy has 0 HP, not -5, and `is_alive` is `False`. Healing 100 only brings HP back to 25, which is `hp_max`. HP always stays between 0 and `hp_max`. Trying to set `is_alive` raises an `AttributeError`, since it is a read-only property.

___________________________________
## Task 2: Dice

`Dice` can create any n-sided dice (n >= 2, default 6). `roll()` returns a random number from 1 to `sides` using `random.randint`. `__call__` makes the object callable, so `d20()` does the same as `d20.roll()`. `__repr__` shows the dice as `Dice(sides=20)`.

The task text says `__call__` should allow rolling with `d20.roll()`, but `roll()` works without `__call__`. We therefore interpret it as making `d20()` possible, which is what `__call__` does.

In [4]:
random.seed(42)  # Fixed seed so the results are reproducible

d6 = Dice()
d20 = Dice(sides=20)
print(d6, d20)
print([d6.roll() for _ in range(10)])
print([d20() for _ in range(10)])

try:
    Dice(sides=1)
except ValueError as err:
    print(f"ValueError: {err}")

Dice(sides=6) Dice(sides=20)
[6, 1, 1, 6, 3, 2, 2, 2, 6, 1]
[18, 3, 19, 14, 2, 1, 3, 7, 8, 17]
ValueError: A dice must have at least 2 sides


**Result:** The default dice has 6 sides, and all rolls stay between 1 and `sides`. Calling `d20()` works the same as `d20.roll()`. Creating a dice with 1 side raises a `ValueError`, so the rule n >= 2 is enforced.

--------------------
## Task 3: Weapon

A `Weapon` has a name and a `Dice`. `roll_damage()` rolls the dice. If a character has a weapon, `attack()` uses the weapon damage instead of the base damage.

In [5]:
random.seed(42)

sword = Weapon("Sword", Dice(sides=12))
armed = Character("Armed", weapon=sword)
unarmed = Character("Unarmed")

for attacker in [armed, unarmed]:
    damages = []
    for _ in range(5):
        target = Character("Target")
        attacker.attack(target)
        damages.append(target.hp_max - target.hp)
    print(f"{attacker.name}: {damages}")

Armed: [11, 2, 1, 12, 5]
Unarmed: [10, 10, 10, 10, 10]


**Result:** With a sword the damage varies between 1 and 12 depending on the dice roll. Without a weapon the character always deals its fixed base damage of 10.

------
## Task 4: Warrior and Mage

`Warrior` and `Mage` are subclasses of `Character`. A `Warrior` has armor that reduces incoming damage. A `Mage` uses 10 mana for a magic attack with double damage. Without enough mana it uses a normal attack and restores 5 mana.

In [6]:
warrior = Warrior("Warrior")
mage = Mage("Mage")

for i in range(5):
    mage.attack(warrior)
    print(warrior.hp, mage.mana)

85 20
70 10
55 0
50 5
45 10


**Result:** The first three attacks are magic: 20 damage minus 5 armor gives 15. Then the mana is too low, so the mage uses normal attacks: 10 minus 5 armor gives 5 damage, and the mage gets 5 mana back each time.

In [7]:
random.seed(42)

mage = Mage("Mage", weapon=Weapon("Staff", Dice(sides=4)))

for _ in range(5):
    target = Character("Target")
    mana_before = mage.mana
    mage.attack(target)
    print(f"Mana {mana_before} -> {mage.mana}, damage dealt: {target.hp_max - target.hp}")

Mana 30 -> 20, damage dealt: 20
Mana 20 -> 10, damage dealt: 20
Mana 10 -> 0, damage dealt: 20
Mana 0 -> 5, damage dealt: 1
Mana 5 -> 10, damage dealt: 1


**Result:** With mana the Mage deals 20 damage (2 x base damage), even though it holds a staff. When mana runs out, it attacks with the staff instead (d4, so at most 4 damage, not the base damage of 10) and restores 5 mana per attack.

-------------
## Task 5: Battle

`Battle` manages turn-based combat between a hero and an enemy. In `play_round()` the hero attacks first, and the enemy strikes back if it is still alive. `simulate_battle()` runs rounds in a while loop until one character dies, and returns the winner.

The damage shown is measured as HP before minus HP after the attack, so it includes the Warrior's armor reduction. We also added a `max_rounds` limit (default 100), since two characters with high armor and low damage could otherwise fight forever.

In [8]:
random.seed(42)

hero = Warrior("Aragorn", weapon=Weapon("Sword", Dice(sides=12)))
enemy = Mage("Saruman")

battle = Battle(hero, enemy)
winner = battle.simulate_battle()


Round 1
Aragorn hits Saruman for 11 damage (89 HP left)
Saruman hits Aragorn for 15 damage (85 HP left)

Round 2
Aragorn hits Saruman for 2 damage (87 HP left)
Saruman hits Aragorn for 15 damage (70 HP left)

Round 3
Aragorn hits Saruman for 1 damage (86 HP left)
Saruman hits Aragorn for 15 damage (55 HP left)

Round 4
Aragorn hits Saruman for 12 damage (74 HP left)
Saruman hits Aragorn for 5 damage (50 HP left)

Round 5
Aragorn hits Saruman for 5 damage (69 HP left)
Saruman hits Aragorn for 5 damage (45 HP left)

Round 6
Aragorn hits Saruman for 4 damage (65 HP left)
Saruman hits Aragorn for 15 damage (30 HP left)

Round 7
Aragorn hits Saruman for 4 damage (61 HP left)
Saruman hits Aragorn for 5 damage (25 HP left)

Round 8
Aragorn hits Saruman for 3 damage (58 HP left)
Saruman hits Aragorn for 5 damage (20 HP left)

Round 9
Aragorn hits Saruman for 12 damage (46 HP left)
Saruman hits Aragorn for 15 damage (5 HP left)

Round 10
Aragorn hits Saruman for 2 damage (44 HP left)
Saruman h

**Result:** Saruman wins after 10 rounds with 44 HP left. The Mage's damage follows a fixed pattern: magic attacks deal 20 minus 5 armor = 15, and base attacks deal 10 minus 5 = 5. Since the Mage has no weapon, this pattern is the same every time, and the Warrior always dies in round 10. Aragorn would need an average of 10 damage per hit to win, but a d12 sword only averages 6.5.

In [9]:
tank_a = Warrior("Tank A", damage=5, armor=5)
tank_b = Warrior("Tank B", damage=5, armor=5)

draw = Battle(tank_a, tank_b, max_rounds=3).simulate_battle()


Round 1
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 2
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 3
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Draw: max rounds reached


**Result:** Both Warriors deal 5 damage and have 5 armor, so every attack deals 0 damage. Without `max_rounds` the while loop would never end. With the limit, the battle stops after 3 rounds and `simulate_battle()` returns `None` to indicate a draw.

___________________________________
## Battles between different character types

First we show two full battles, without and with weapons. Then we use `verbose=False` in `Battle` to run 1000 silent battles per matchup and compare win rates.

In [10]:
hero = Character("Knight")
enemy = Warrior("Conan")
winner = Battle(hero, enemy).simulate_battle()


Round 1
Knight hits Conan for 5 damage (95 HP left)
Conan hits Knight for 10 damage (90 HP left)

Round 2
Knight hits Conan for 5 damage (90 HP left)
Conan hits Knight for 10 damage (80 HP left)

Round 3
Knight hits Conan for 5 damage (85 HP left)
Conan hits Knight for 10 damage (70 HP left)

Round 4
Knight hits Conan for 5 damage (80 HP left)
Conan hits Knight for 10 damage (60 HP left)

Round 5
Knight hits Conan for 5 damage (75 HP left)
Conan hits Knight for 10 damage (50 HP left)

Round 6
Knight hits Conan for 5 damage (70 HP left)
Conan hits Knight for 10 damage (40 HP left)

Round 7
Knight hits Conan for 5 damage (65 HP left)
Conan hits Knight for 10 damage (30 HP left)

Round 8
Knight hits Conan for 5 damage (60 HP left)
Conan hits Knight for 10 damage (20 HP left)

Round 9
Knight hits Conan for 5 damage (55 HP left)
Conan hits Knight for 10 damage (10 HP left)

Round 10
Knight hits Conan for 5 damage (50 HP left)
Conan hits Knight for 10 damage (0 HP left)

Winner: Conan


**Result:** Without weapons the battle is fully deterministic. The Knight deals 10 minus 5 armor = 5 damage per hit, while Conan deals 10. Conan wins in round 10 with 50 HP left.

In [11]:
random.seed(42)

hero = Character("Knight", weapon=Weapon("Sword", Dice(sides=12)))
enemy = Warrior("Conan", weapon=Weapon("Sword", Dice(sides=12)))
winner = Battle(hero, enemy).simulate_battle()


Round 1
Knight hits Conan for 6 damage (94 HP left)
Conan hits Knight for 2 damage (98 HP left)

Round 2
Knight hits Conan for 0 damage (94 HP left)
Conan hits Knight for 12 damage (86 HP left)

Round 3
Knight hits Conan for 0 damage (94 HP left)
Conan hits Knight for 4 damage (82 HP left)

Round 4
Knight hits Conan for 0 damage (94 HP left)
Conan hits Knight for 3 damage (79 HP left)

Round 5
Knight hits Conan for 7 damage (87 HP left)
Conan hits Knight for 2 damage (77 HP left)

Round 6
Knight hits Conan for 6 damage (81 HP left)
Conan hits Knight for 12 damage (65 HP left)

Round 7
Knight hits Conan for 4 damage (77 HP left)
Conan hits Knight for 2 damage (63 HP left)

Round 8
Knight hits Conan for 5 damage (72 HP left)
Conan hits Knight for 7 damage (56 HP left)

Round 9
Knight hits Conan for 0 damage (72 HP left)
Conan hits Knight for 1 damage (55 HP left)

Round 10
Knight hits Conan for 0 damage (72 HP left)
Conan hits Knight for 4 damage (51 HP left)

Round 11
Knight hits Conan

**Result:** Conan wins in round 19 with 44 HP left. The Knight's sword often deals 0 damage, since any roll of 5 or lower is fully blocked by the armor.

In [12]:
def win_rates(hero_class: type, enemy_class: type, weapons: bool, n: int = 1000) -> tuple[float, float, float]:
    """
    Run n battles between two character types and count the results.

    Args:
        hero_class (type): Class of the hero, e.g. Warrior.
        enemy_class (type): Class of the enemy, e.g. Mage.
        weapons (bool): If True, both characters get a d12 sword.
        n (int): Number of battles (default = 1000).

    Returns:
        tuple[float, float, float]: Share of hero wins, enemy wins and draws.
    """
    hero_wins, enemy_wins = 0, 0
    for _ in range(n):
        hero_weapon = Weapon("Sword", Dice(sides=12)) if weapons else None
        enemy_weapon = Weapon("Sword", Dice(sides=12)) if weapons else None
        hero = hero_class("Hero", weapon=hero_weapon)
        enemy = enemy_class("Enemy", weapon=enemy_weapon)
        winner = Battle(hero, enemy, verbose=False).simulate_battle()
        if winner is hero:
            hero_wins += 1
        elif winner is enemy:
            enemy_wins += 1
    return hero_wins / n, enemy_wins / n, (n - hero_wins - enemy_wins) / n


random.seed(42)
matchups = [(Character, Character), (Character, Warrior), (Character, Mage), (Warrior, Mage), (Mage, Warrior)]

print(f"{'Hero':<11}{'Enemy':<11}{'Weapons':<10}{'Hero wins':>10}{'Enemy wins':>12}{'Draws':>8}")
for hero_class, enemy_class in matchups:
    for weapons in [False, True]:
        hero, enemy, draw = win_rates(hero_class, enemy_class, weapons)
        print(f"{hero_class.__name__:<11}{enemy_class.__name__:<11}{str(weapons):<10}{hero:>10.1%}{enemy:>12.1%}{draw:>8.1%}")

Hero       Enemy      Weapons    Hero wins  Enemy wins   Draws
Character  Character  False         100.0%        0.0%    0.0%
Character  Character  True           53.6%       46.4%    0.0%
Character  Warrior    False           0.0%      100.0%    0.0%
Character  Warrior    True            0.0%      100.0%    0.0%
Character  Mage       False           0.0%      100.0%    0.0%
Character  Mage       True            0.0%      100.0%    0.0%
Warrior    Mage       False         100.0%        0.0%    0.0%
Warrior    Mage       True            9.6%       90.4%    0.0%
Mage       Warrior    False         100.0%        0.0%    0.0%
Mage       Warrior    True           95.7%        4.3%    0.0%


**Result:** Four findings stand out:

1. **Without weapons every battle is deterministic.** No dice are rolled, so the same matchup always gives the same winner.
2. **The hero has a first mover advantage.** Two equal Characters with swords give the hero 53.6 % wins. Without weapons, Warrior vs Mage is won by whoever attacks first, since both would die in round 10.
3. **Armor neutralises the sword.** A Character never beats a Warrior, because a d12 sword deals only about 2.3 damage per hit on average after 5 armor.
4. **The sword is a downgrade.** A d12 averages 6.5 damage, which is less than the base damage of 10. The Warrior goes from 100 % to 9.6 % wins against the Mage when both get swords, while the Mage is barely affected, since its magic attacks ignore the weapon.

___________________________________
## Discussion

### Object-oriented design
Each class models one concept and lives in its own module, apart from `Warrior` and `Mage`, which extend `character.py` as the task describes. The project uses the three main ideas from the lectures:

- **Encapsulation:** HP is stored in the protected attribute `_hp`, read through the `hp` property and only changed by the character's own `take_damage()` and `heal()`. This guarantees that HP always stays between 0 and `hp_max`.
- **Inheritance:** `Warrior` and `Mage` reuse everything in `Character` and only override what makes them different, `take_damage()` and `attack()`.
- **Composition:** A `Character` has a `Weapon`, which has a `Dice`. Changing the dice changes the damage without touching the character classes.

### Design choices
- `Battle` measures damage as HP before minus HP after the attack. This shows the actual damage after armor, without `Battle` needing to know how each subclass calculates damage.
- `max_rounds` stops battles that would otherwise never end, as shown with the two Warriors.
- `verbose=False` lets us run 1000 battles per matchup without printing every round.
- `random.seed(42)` makes all results reproducible, so the text in the notebook matches the output.

### Game balance
The simulations show that the game is not balanced. The d12 sword averages 6.5 damage, which is lower than the base damage of 10, so equipping a weapon makes a character weaker. Armor of 5 makes the Warrior almost immune to the sword, and the Mage is barely affected by weapons since its magic ignores them. When two characters are otherwise equal, the hero wins more often because it attacks first.

### Possible improvements
- Balance the weapons, for example with a d20 or a damage bonus, so that a weapon is an upgrade.
- Let the starting attacker be chosen randomly to remove the first mover advantage.
- Make `mana` protected with a property, in the same way as `_hp`.
- Expand `items.py` with other items, such as healing potions that use `heal()`.